In [1]:
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from dotenv import load_dotenv
import json

sys.path.append(os.path.abspath("../.."))
from scraper import fetch_website_contents, fetch_website_links

In [3]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:4]}")
else:
    print("GROQ API Key not set (and this is optional)")

GROQ API Key exists and begins gsk_


In [18]:
groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(
    base_url=groq_url,
    api_key=groq_api_key
)

MODEL = "openai/gpt-oss-120b"

In [5]:
links = fetch_website_links("https://ollama.com")
links

['/',
 '/search',
 '/docs',
 '/pricing',
 '/signin',
 '/download',
 '/search',
 '/download',
 '/docs',
 '/pricing',
 '/signin',
 'https://docs.ollama.com/integrations/openclaw',
 '/download/windows',
 '/download',
 'https://docs.ollama.com/integrations',
 '/signup',
 '/upgrade?plan=pro&interval=year',
 '/upgrade?plan=pro',
 '/upgrade?plan=max',
 '/download',
 '/download',
 '/blog',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'mailto:hello@ollama.com',
 '/privacy',
 '/terms',
 '/blog',
 '/download',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'https://lu.ma/ollama',
 '/privacy',
 '/terms']

## First we are using llama3.2:3b model to figure out which links are relevant for broucher.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://ollama.com"))


Here is the list of links on the website https://ollama.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

/
/search
/docs
/pricing
/signin
/download
/search
/download
/docs
/pricing
/signin
https://docs.ollama.com/integrations/openclaw
/download/windows
/download
https://docs.ollama.com/integrations
/signup
/upgrade?plan=pro&interval=year
/upgrade?plan=pro
/upgrade?plan=max
/download
/download
/blog
https://docs.ollama.com
https://github.com/ollama/ollama
https://discord.com/invite/ollama
https://twitter.com/ollama
mailto:hello@ollama.com
/privacy
/terms
/blog
/download
https://docs.ollama.com
https://github.com/ollama/ollama
h

In [19]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
select_relevant_links("https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 8 relevant links


{'links': [{'type': 'forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'product‑platform', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'status', 'url': 'https://status.huggingface.co/'},
  {'type': 'fast‑inference', 'url': 'https://fast.hf.co'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        if link['url'].startswith("https://"):
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co/"))

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
bytedance-research/Lance
Updated
1 day ago
•
739
•
548
Supertone/supertonic-3
Updated
3 days ago
•
35k
•
523
openbmb/MiniCPM-V-4.6
Updated
2 days ago
•
196k
•
852
SulphurAI/Sulphur-2-base
Updated
4 days ago
•
1.2M
•
1.22k
unsloth/Qwen3.6-27B-MTP-GGUF
Updated
1 day ago
•
478k
•
370
Bro

In [22]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:7_000] # Truncate if more than 7,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("edwarddonner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 6 relevant links


'\nYou are looking at a company called: edwarddonner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 20

In [24]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 6 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your gateway to open, collaborative, and ethical AI.*

---

## 📌 Who We Are  

Hugging Face is the **collaboration platform for the global machine‑learning community**.  
- **Open‑source first:** Host and share unlimited public models, datasets, and applications.  
- **Community‑driven:** Over **2 million models**, **500 k+ datasets**, and **1 million+ AI apps (Spaces)** are created, discovered, and improved every day by researchers, engineers, and hobbyists.  
- **Ethical AI:** We empower the next generation of ML engineers and scientists to build an *open and responsible* AI future together.  

> “Hugging Face is the collaboration platform for the machine learning community… to learn, collaborate and share their work to build an open and ethical AI future together.” – HF Brand Bio

---

## 🌐 What We Offer  

| Product | Core Benefits | Typical Users |
|---------|---------------|---------------|
| **Models Hub** | Browse 2 M+ pretrained models, fine‑tune or deploy with a single click. | Researchers, developers, enterprises |
| **Datasets Hub** | 500 k+ high‑quality datasets, ready for training and evaluation. | Data scientists, academia |
| **Spaces** | Host interactive demos & AI applications (e.g., image‑to‑3D, TTS, video generation). | Creators, product teams, educators |
| **Inference Endpoints & Providers** | Scalable, low‑latency serving for LLMs and vision models; managed or self‑hosted. | SaaS platforms, fintech, health tech |
| **Enterprise Solutions** | Private hubs, compliance, SSO, dedicated support, and custom SLAs. | Fortune 500, government, regulated industries |
| **Hugging Face PRO & Enterprise Support** | Priority assistance, advanced analytics, and roadmap influence. | Teams scaling AI production |
| **Storage Buckets** | Secure, versioned storage for large model artifacts and datasets. | MLOps pipelines, research labs |

---

## 🤝 Community & Ecosystem  

- **Vibrant Collaboration:** Forums, Discord, GitHub, and a daily “Blog & Papers” feed keep knowledge flowing.  
- **Open‑source Libraries:** Transformers, Diffusers, Datasets, Accelerate, and more—used in **90 % of the world’s top AI papers**.  
- **Partnerships:** Joint projects with leading labs (e.g., Bytedance Research, OpenBMB) and industry‑specific solutions for NLP, CV, Audio, RL, and Diffusion.  
- **Learning Resources:** “Learn” hub, tutorials, and the **Hugging Face Universe** of brand assets for educators and marketers.  

---

## 🏢 Who Trusts Hugging Face  

- **Tech giants & startups** building LLM‑powered products.  
- **Enterprises** leveraging private hubs for secure model sharing and compliance.  
- **Research institutions** publishing state‑of‑the‑art models and datasets.  
- **Developers worldwide** who host Spaces ranging from 3‑D generation (Pixal3D) to multilingual TTS (Supertonic 3).  

*Exact customer names are disclosed on request or via case‑study pages.*

---

## 🚀 Why Invest in Hugging Face  

- **Fast‑growing market:** AI model sharing is becoming the de‑facto infrastructure for LLM and foundation model adoption.  
- **Monetizable stack:** Enterprise subscriptions, inference billing, and professional services generate recurring revenue.  
- **Defensible moat:** Network effects of a massive open‑source ecosystem and a trusted brand for ethical AI.  
- **World‑class talent:** A science team that pushes the frontier of transformer research and a community of over a million contributors.  

---

## 👩‍💻 Careers & Culture  

**Culture** – Collaborative, transparent, and purpose‑driven. We celebrate open science, diversity, and continuous learning.  

**Opportunities** – Roles across:  

- **Research & Engineering** (ML research, model scaling, inference optimization)  
- **Product & Design** (UX for Spaces, platform features)  
- **Customer Success & Enterprise** (solution architecture, support)  
- **Community & Partnerships** (developer advocacy, open‑source program)  

Visit our **Careers** page for current openings and to join a team shaping the future of AI.  

---

## 📞 Get in Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, GitHub, Twitter, LinkedIn  
- **Press & Media:** press@huggingface.co  
- **Enterprise Inquiries:** sales@huggingface.co  

---

*Hugging Face – Empowering the world to build, share, and responsibly deploy AI.*

In [43]:
def stream_brochure(url, company_name):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [37]:
stream_brochure("https://huggingface.co", "Hugging Face")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links


**Hugging Face – The AI Community Building the Future**  

---

### 🌟 Company Overview  
Hugging Face is the world’s leading collaboration platform for the machine‑learning ecosystem. Through the **Hugging Face Hub** it hosts 2 million+ open‑source models, 500 k+ datasets and 1 million+ AI applications (Spaces). The platform brings together researchers, engineers, startups, enterprises and hobbyists to **share, discover, experiment and deploy** state‑of‑the‑art AI responsibly.

---

### 🎯 Mission & Vision  
> “Empower the next generation of machine‑learning engineers, scientists, and end‑users to learn, collaborate and share their work to build an **open and ethical AI future** together.”  

Hugging Face believes that openness, reproducibility and community‑driven innovation are the foundations for trustworthy AI.

---

### 🚀 Core Products & Solutions  

| Product | What it does | Typical users |
|--------|--------------|----------------|
| **Models** | Browse, host and fine‑tune 2 M+ pre‑trained models (LLMs, vision, audio, RL, diffusion, etc.) | Researchers, developers, product teams |
| **Datasets** | Access 500 k+ curated datasets for training and evaluation | Data scientists, academia |
| **Spaces** | Deploy interactive demos and AI apps with zero‑code or custom code | Creators, startups, educators |
| **Inference Endpoints & Providers** | Scalable, low‑latency model serving on the cloud or on‑prem | Enterprises needing production‑grade AI |
| **Buckets (Storage)** | Secure object storage for large model weights and data assets | Teams handling big‑scale training pipelines |
| **Hugging Face PRO & Enterprise** | Premium support, private repositories, compliance tools, SLA guarantees | Large organizations, regulated industries |
| **HuggingChat** | Open‑source conversational AI interface powered by community models | End‑users, developers building chat experiences |

---

### 🤝 Community & Culture  

* **Open‑source at heart** – Every model, dataset and Space is publicly viewable and remixable unless a user opts for a private repo.  
* **Collaboration‑first** – The Hub acts as a Git‑like versioned store for ML assets, enabling reproducible research and rapid iteration.  
* **Ethical AI** – The brand narrative stresses an “open and ethical AI future,” and the company runs regular community discussions on bias, safety and governance.  
* **Global, inclusive** – Multilingual resources (e.g., multilingual TTS, Vietnamese OCR) and a vibrant Discord, Forum and GitHub presence make the community truly worldwide.  
* **Fast‑growing talent pool** – The science team pushes the frontier of model architecture, while product teams focus on developer experience and enterprise readiness.  

---

### 📈 Customers & Impact  

* **Enterprise adopters** use Hugging Face Enterprise for secure model hosting, compliance‑ready pipelines and dedicated support.  
* **Start‑ups & developers** launch AI‑driven products directly from Spaces (e.g., image‑to‑3D generation, video synthesis, multilingual TTS).  
* **Academic & research labs** publish and benchmark models such as MiniCPM‑V‑4.6, Sulphur‑2‑base and Qwen3.6‑27B, gaining visibility across the community.  
* **Large tech partners** (e.g., Bytedance research) contribute flagship models that become community standards.  

---

### 👩‍💻 Careers & Opportunities  

Hugging Face is actively hiring across engineering, research, product, design, sales and community roles. The **Careers** page lists current openings—candidates are encouraged to bring a passion for open‑source, AI ethics, and collaborative product building.  

*Roles you might find:*  

* Machine‑learning research scientists  
* Software engineers (backend, inference, UI)  
* DevOps / Cloud infrastructure engineers  
* Community program managers  
* Product managers for enterprise and developer tooling  
* UX/UI designers focused on ML workflows  

The company promotes a remote‑first, inclusive culture where every voice can influence the direction of the AI ecosystem.

---

### 📚 Learn More & Get Involved  

* **Website:** https://huggingface.co  
* **Docs & Tutorials:** Comprehensive guides for models, datasets, Spaces, and deployment.  
* **Community:** Discord, Forum, GitHub, and regular blog posts covering research breakthroughs, ethics, and case studies.  
* **Brand Assets:** Official logos, color palette (#FFD21E, #FF9D00, #6B7280) and brand guidelines are available for partners.  

---

**Join the movement.** Whether you’re looking to experiment with the latest LLM, deploy production‑grade AI, or contribute to an open, responsible future of machine learning, Hugging Face provides the tools, community and platform to make it happen.

### Now we are going to make UI for this website broucher generator using Gradio.

#### This is week-2, day-2 exercise :)

In [38]:
import gradio as gr

In [ ]:
def stream_brochure_for_gr(url, company_name):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [52]:
message_input = gr.Textbox(label="Your website link:", info="Enter the URL of the website for which you want to generate a brochure", lines=1)
message_input2 = gr.Textbox(label="Your company name:", info="Enter the name of the company", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure_for_gr,
    title="Website brochure generator", 
    inputs=[message_input, message_input2], 
    outputs=[message_output], 
    examples=[
        ["https://huggingface.co", "Hugging Face"],
        ["https://ollama.com", "Ollama"],
        ["https://udemy.com", "Udemy"],
        ["https://edwarddonner.com", "Edward Donner"]
        ], 
    flagging_mode="never"
    )

view.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 5 relevant links
